# Chapter 7 — Quantization & Export

**Goal:** Compress our fine-tuned model so it runs efficiently on a laptop CPU.

Quantization reduces weight precision (FP32 → INT8 or INT4), cutting memory 2-4×. For a 135M model: ~540MB → under 100MB.

**Critical:** LoRA adapters must be merged into the base model in full precision FIRST, then the merged model is quantized. Merging into an already-quantized model corrupts the weights.

## 7.1 Load Adapter + Merge in FP32

Step 1: Load the base model at full precision, load the LoRA adapter, merge into a single model.

In [ ]:
import torch, os, time, shutil
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

BASE_MODEL = "HuggingFaceTB/SmolLM-135M"
device = "cuda" if torch.cuda.is_available() else "cpu"

# Try common adapter locations
ADAPTER_PATHS = [
    "../02-classification/adapter",
    "../02-classification/chapters/02-classification/adapter",
    "./chapters/02-classification/adapter",
]
ADAPTER_PATH = None
for p in ADAPTER_PATHS:
    if os.path.exists(p):
        ADAPTER_PATH = p
        break

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load at full precision, merge adapter, then save merged model
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, torch_dtype=torch.float32,
    device_map="auto" if device == "cuda" else None,
)

if ADAPTER_PATH:
    model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
    model = model.merge_and_unload()  # bake LoRA into base weights
    print(f"Merged adapter: {ADAPTER_PATH}")
else:
    model = base_model
    print("No adapter found — using raw base model")

# Save the merged FP32 model
MERGED_PATH = "./merged-model"
model.save_pretrained(MERGED_PATH)
tokenizer.save_pretrained(MERGED_PATH)

def dir_size(path):
    total = 0
    for dirpath, _, filenames in os.walk(path):
        for f in filenames:
            total += os.path.getsize(os.path.join(dirpath, f))
    return total / (1024 * 1024)

fp32_mb = dir_size(MERGED_PATH)
print(f"Merged FP32 model: {fp32_mb:.0f} MB  |  saved to {MERGED_PATH}")

## 7.2 Baseline: FP32 Accuracy & Speed

In [ ]:
# Load back the merged FP32 model
model_fp32 = AutoModelForCausalLM.from_pretrained(MERGED_PATH, torch_dtype=torch.float32).to("cpu")
model_fp32.eval()

VALID_LABELS = {"bug", "feature_request", "praise", "question"}

def classify(model_obj, text):
    prompt = f"Classify: {text}\nLabel:"
    inputs = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        outputs = model_obj.generate(**inputs, max_new_tokens=3, do_sample=False,
                                     pad_token_id=tokenizer.eos_token_id,
                                     eos_token_id=tokenizer.eos_token_id)
    full = tokenizer.decode(outputs[0], skip_special_tokens=True)
    if "Label:" in full:
        raw = full.split("Label:")[-1].strip()
        for label in VALID_LABELS:
            if raw.startswith(label):
                return label
    return "???"

test_texts = [
    ("the app keeps crashing when I try to upload a photo", "bug"),
    ("would be great if we could have dark mode", "feature_request"),
    ("this is honestly the best note-taking app I've used", "praise"),
    ("how do I reset my password from the mobile app", "question"),
]

# Accuracy
correct = sum(1 for text, exp in test_texts if classify(model_fp32, text) == exp)
print(f"FP32 accuracy: {correct}/{len(test_texts)}")

# CPU speed
test_prompt = "Classify: the app keeps crashing when I upload photos\nLabel:"
inputs = tokenizer(test_prompt, return_tensors="pt")
runs = 5
with torch.no_grad():
    _ = model_fp32.generate(**inputs, max_new_tokens=5, do_sample=False, pad_token_id=tokenizer.eos_token_id)
start = time.time()
with torch.no_grad():
    for _ in range(runs):
        _ = model_fp32.generate(**inputs, max_new_tokens=5, do_sample=False, pad_token_id=tokenizer.eos_token_id)
fp32_time = (time.time() - start) / runs
print(f"FP32 CPU speed: {fp32_time:.2f}s per request")

## 7.3 INT8 Quantization

Load the MERGED model (not adapter-on-quantized) with 8-bit quantization. Each weight: 32 bits → 8 bits.

In [ ]:
from transformers import BitsAndBytesConfig

quant_8bit = BitsAndBytesConfig(load_in_8bit=True, llm_int8_threshold=6.0)

model_8bit = AutoModelForCausalLM.from_pretrained(
    MERGED_PATH,
    quantization_config=quant_8bit,
    device_map="auto" if device == "cuda" else None,
)
model_8bit.eval()

# Accuracy
correct = sum(1 for text, exp in test_texts if classify(model_8bit, text) == exp)
print(f"INT8 accuracy: {correct}/{len(test_texts)}")

# CPU speed
with torch.no_grad():
    _ = model_8bit.generate(**inputs, max_new_tokens=5, do_sample=False, pad_token_id=tokenizer.eos_token_id)
start = time.time()
with torch.no_grad():
    for _ in range(runs):
        _ = model_8bit.generate(**inputs, max_new_tokens=5, do_sample=False, pad_token_id=tokenizer.eos_token_id)
int8_time = (time.time() - start) / runs
print(f"INT8 CPU speed: {int8_time:.2f}s per request")
print(f"Size: ~{fp32_mb/4:.0f} MB (4× smaller)")

## 7.4 INT4 Quantization

Load the MERGED model with 4-bit NormalFloat quantization. Each weight: 32 bits → 4 bits.

In [ ]:
quant_4bit = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

model_4bit = AutoModelForCausalLM.from_pretrained(
    MERGED_PATH,
    quantization_config=quant_4bit,
    device_map="auto" if device == "cuda" else None,
)
model_4bit.eval()

# Accuracy
correct = sum(1 for text, exp in test_texts if classify(model_4bit, text) == exp)
print(f"INT4 accuracy: {correct}/{len(test_texts)}")

# CPU speed
with torch.no_grad():
    _ = model_4bit.generate(**inputs, max_new_tokens=5, do_sample=False, pad_token_id=tokenizer.eos_token_id)
start = time.time()
with torch.no_grad():
    for _ in range(runs):
        _ = model_4bit.generate(**inputs, max_new_tokens=5, do_sample=False, pad_token_id=tokenizer.eos_token_id)
int4_time = (time.time() - start) / runs
print(f"INT4 CPU speed: {int4_time:.2f}s per request")
print(f"Size: ~{fp32_mb/8:.0f} MB (8× smaller)")

## 7.5 Comparison

The full picture: size, speed, and accuracy across precisions.

In [ ]:
# Gather results from above cells
fp32_acc = sum(1 for text, exp in test_texts if classify(model_fp32, text) == exp)
int8_acc = sum(1 for text, exp in test_texts if classify(model_8bit, text) == exp)
int4_acc = sum(1 for text, exp in test_texts if classify(model_4bit, text) == exp)

print("Quantization comparison — SmolLM 135M:\n")
print(f"  {'Precision':<10} {'Size':>8} {'Speed':>8} {'Accuracy':>10}")
print(f"  {'─'*10} {'─'*8} {'─'*8} {'─'*10}")
print(f"  {'FP32':<10} {fp32_mb:>6.0f} MB {fp32_time:>6.2f}s  {fp32_acc:>6}/{len(test_texts)}")
print(f"  {'INT8':<10} {fp32_mb/4:>6.0f} MB {int8_time:>6.2f}s  {int8_acc:>6}/{len(test_texts)}")
print(f"  {'INT4':<10} {fp32_mb/8:>6.0f} MB {int4_time:>6.2f}s  {int4_acc:>6}/{len(test_texts)}")

print("\nKey insight:")
print("  INT8: near-identical accuracy at 4× smaller — the sweet spot for most use cases")
print("  INT4: 8× smaller, some accuracy loss — fine for simple tasks on constrained devices")
print(f"  A {fp32_mb/8:.0f}MB model runs on any phone, any laptop, even a Raspberry Pi")

## 7.6 The GGUF Path (Production Deployment)

bitsandbytes is for development. Production uses **GGUF** — the format behind llama.cpp, Ollama, and LM Studio. Single file, CPU-only, no Python needed.

```
Merged HF model (FP32)
    ↓ convert_hf_to_gguf.py
FP16 GGUF
    ↓ llama-quantize (q4_K_M, q8_0, etc.)
Quantized GGUF  ← ships to users
```

Common GGUF levels:

| Level | Bits | Best for |
|---|---|---|
| `q8_0` | 8-bit | Desktop, near-lossless |
| `q4_K_M` | 4-bit | Laptop, flagship phone |
| `q4_0` | 4-bit | Budget devices |
| `q2_K` | 2-bit | Extreme compression |

```bash
# Convert to GGUF (requires llama.cpp)
git clone https://github.com/ggerganov/llama.cpp && cd llama.cpp && make
python convert_hf_to_gguf.py ../chapters/07-quantization/merged-model \
    --outtype f16 --outfile smollm-f16.gguf
./llama-quantize smollm-f16.gguf smollm-Q4_K_M.gguf Q4_K_M
./llama-cli -m smollm-Q4_K_M.gguf -p "Classify: app crashes\nLabel:"
```

## 7.7 End-to-End Summary

| Step | Action | Size |
|---|---|---|
| 1. Fine-tune | Train SmolLM + LoRA adapter | 540MB + 2MB adapter |
| 2. Merge | `merge_and_unload()` → bake adapter into weights | 540MB |
| 3. Quantize INT8 | Load merged model with bitsandbytes | ~135MB |
| 4. Quantize INT4 | Load merged model with bitsandbytes NF4 | ~68MB |
| 5. Convert GGUF | llama.cpp conversion | ~70-140MB single file |
| 6. Deploy | Run on CPU with llama.cpp / Ollama | No GPU, no Python |

A task-specialized 135M model at INT4 runs on any device with 2GB+ RAM. That's the goal we set out to achieve.